# Yield Curve Models

This notebook covers parametric and non-parametric yield curve fitting using `bond.curves`.

**Topics**
- Nelson-Siegel model: level, slope, and curvature
- Svensson model: adding a second hump
- Cubic spline: non-parametric interpolation
- Live US Treasury data from FRED
- Forward rate extraction

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/bond/blob/main/notebooks/02_yield_curves.ipynb)

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np
import matplotlib.pyplot as plt
from bond import (
    fit_nelson_siegel, spot_rate,
    fit_svensson, spot_rate_svensson,
    fit_cubic_spline, forward_rate, plot_curve,
)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('bond.curves loaded ✓')

## Sample Market Data

We use a stylized US Treasury yield curve. Replace with `fetch_treasury_yields()` for live data.

In [ ]:
# Typical upward-sloping curve
maturities = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]
yields_normal = [0.0530, 0.0525, 0.0510, 0.0490, 0.0475, 0.0455, 0.0445, 0.0440, 0.0435, 0.0430]

# Inverted curve
yields_inverted = [0.0560, 0.0555, 0.0545, 0.0530, 0.0510, 0.0480, 0.0460, 0.0440, 0.0420, 0.0410]

fig, ax = plt.subplots()
ax.plot(maturities, np.array(yields_normal)*100, 'o-', lw=2, label='Normal curve')
ax.plot(maturities, np.array(yields_inverted)*100, 's--', lw=2, label='Inverted curve')
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Yield (%)')
ax.set_title('Sample Yield Curves')
ax.legend()
plt.tight_layout()
plt.show()

## Nelson-Siegel Model

The Nelson-Siegel (1987) model parameterizes the yield curve with four parameters:

$$r(t) = \beta_0 + \beta_1 \frac{1-e^{-t/\tau}}{t/\tau} + \beta_2 \left(\frac{1-e^{-t/\tau}}{t/\tau} - e^{-t/\tau}\right)$$

- $\beta_0$: long-run level
- $\beta_1$: slope (short rate offset)
- $\beta_2$: curvature / hump
- $\tau$: decay factor (location of hump)

In [ ]:
ns_params = fit_nelson_siegel(maturities, yields_normal)
b0, b1, b2, tau = ns_params
print(f'Nelson-Siegel parameters:')
print(f'  β₀ (level):    {b0:.4%}')
print(f'  β₁ (slope):    {b1:.4%}')
print(f'  β₂ (curvature):{b2:.4%}')
print(f'  τ  (decay):    {tau:.4f}')

t = np.linspace(0.1, 30, 300)
fitted = spot_rate(t, *ns_params)

print(f'\nFit quality:')
for m, y in zip(maturities, yields_normal):
    f = spot_rate(m, *ns_params)
    print(f'  {m:5.2f}y  market={y:.4%}  fitted={f:.4%}  error={abs(f-y)*1e4:.1f}bp')

In [ ]:
plot_curve(ns_params, model='ns', market_mats=maturities,
           market_yields=yields_normal, label='Nelson-Siegel fit')

## Svensson Model

The Svensson (1994) extension adds a second hump term, giving more flexibility for curves with two peaks (e.g. a butterfly shape):

$$r(t) = \beta_0 + \beta_1 L_1(t) + \beta_2 H_1(t) + \beta_3 H_2(t)$$

In [ ]:
sv_params = fit_svensson(maturities, yields_normal)
b0, b1, b2, b3, tau1, tau2 = sv_params
print(f'Svensson parameters:')
print(f'  β₀={b0:.4%}  β₁={b1:.4%}  β₂={b2:.4%}  β₃={b3:.4%}')
print(f'  τ₁={tau1:.4f}  τ₂={tau2:.4f}')

fig, ax = plt.subplots()
t = np.linspace(0.1, 30, 300)
ax.plot(t, spot_rate(t, *ns_params)*100, lw=2, label='Nelson-Siegel')
ax.plot(t, spot_rate_svensson(t, *sv_params)*100, lw=2, ls='--', label='Svensson')
ax.scatter(maturities, np.array(yields_normal)*100, color='black', zorder=5, s=40, label='Market')
ax.set_xlabel('Maturity (years)'); ax.set_ylabel('Yield (%)')
ax.set_title('Nelson-Siegel vs Svensson')
ax.legend()
plt.tight_layout()
plt.show()

## Cubic Spline (Non-Parametric)

A natural cubic spline passes exactly through each market data point and is smooth (continuous second derivative). It is more flexible than parametric models but can oscillate between knots.

In [ ]:
cs = fit_cubic_spline(maturities, yields_normal)

fig, ax = plt.subplots()
t = np.linspace(0.25, 30, 300)
ax.plot(t, spot_rate(t, *ns_params)*100, lw=2, label='Nelson-Siegel')
ax.plot(t, cs(t)*100, lw=2, ls=':', label='Cubic Spline')
ax.scatter(maturities, np.array(yields_normal)*100, color='black', zorder=5, s=40, label='Market')
ax.set_xlabel('Maturity (years)'); ax.set_ylabel('Yield (%)')
ax.set_title('Parametric vs Non-Parametric Curve Fitting')
ax.legend()
plt.tight_layout()
plt.show()

## Forward Rates

The instantaneous forward rate is implied by the spot curve:

$$f(t_1, t_2) = \frac{r(t_2) \cdot t_2 - r(t_1) \cdot t_1}{t_2 - t_1}$$

Forward rates encode market expectations of future short rates (under the expectations hypothesis).

In [ ]:
spot_fn = lambda t: float(spot_rate(t, *ns_params))

t_grid = np.linspace(0.5, 29.5, 100)
fwd_rates = [forward_rate(t, t+0.5, spot_fn) for t in t_grid]
spot_rates = [spot_fn(t) for t in t_grid]

fig, ax = plt.subplots()
ax.plot(t_grid, np.array(spot_rates)*100, lw=2, label='Spot rate (NS)')
ax.plot(t_grid, np.array(fwd_rates)*100, lw=2, ls='--', label='6m forward rate')
ax.set_xlabel('Maturity (years)'); ax.set_ylabel('Rate (%)')
ax.set_title('Spot and Forward Rates from Nelson-Siegel Curve')
ax.legend()
plt.tight_layout()
plt.show()

# Key forward rates
print('Key forward rates:')
for t1, t2 in [(0,1),(1,2),(2,5),(5,10),(10,30)]:
    t1_ = max(t1, 0.001)
    f = forward_rate(t1_, t2, spot_fn)
    print(f'  f({t1}y, {t2}y) = {f:.4%}')

## Live Treasury Data (FRED)

Uncomment the cell below to fetch live US Treasury par yields from FRED.

In [ ]:
# from bond import fetch_treasury_yields
# mats, ylds = fetch_treasury_yields()  # get a free API key at fred.stlouisfed.org
# if mats:
#     params = fit_nelson_siegel(mats, ylds)
#     plot_curve(params, model='ns', market_mats=mats, market_yields=ylds,
#                label='Nelson-Siegel (live Treasury)')
# else:
#     print('No data returned. Check FRED API key.')
print('Uncomment the above cell to use live FRED data.')